# Future Customer Value Prediction

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from pathlib import Path
from urllib.parse import quote_plus
from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor

In [ ]:
server = os.getenv("SQL_SERVER", "localhost")
database = "EcommerceAnalysis"

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

encoded_connection_string = quote_plus(connection_string)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={encoded_connection_string}"
)

In [ ]:
sql_path = Path("05_customer_value_dataset.sql")

customer_query = sql_path.read_text(encoding="utf-8")

df = pd.read_sql(customer_query, engine)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

## 2. Target Analysis

In [ ]:
purchase_distribution = (
    df["future_purchase"]
    .value_counts()
    .sort_index()
)

purchase_distribution

In [ ]:
purchase_percentage = (
    df["future_purchase"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

purchase_percentage.round(2)

## 3. Classification Model Preparation

In [ ]:
y = df["future_purchase"]

feature_columns = [
    "recency_days",
    "frequency",
    "monetary",
    "avg_order_value",
    "avg_items_per_order",
    "category_diversity",
    "favorite_category_share",
    "purchase_frequency",
    "effective_discount_rate",
    "returned_order_rate",
    "average_rating",
    "review_rate"
]

X = df[feature_columns].copy()

### Feature Selection Notes

The following columns were excluded from the initial classification feature set:

- `customer_id`: Identifier only; it does not represent customer behavior.
- `favorite_category`: Categorical feature; excluded from the initial numeric baseline model and may be evaluated later using encoding.
- `future_purchase`: Classification target variable.
- `future_spend`: Future-period information; including it would cause target leakage.

In [ ]:
X.shape, y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
imputer = SimpleImputer(strategy="median")

X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

X_train_imputed[["average_rating"]] = imputer.fit_transform(
    X_train[["average_rating"]]
)

X_test_imputed[["average_rating"]] = imputer.transform(
    X_test[["average_rating"]]
)

## 4. Classification Modeling

### Baseline Model

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train_imputed, y_train)

y_pred_dummy = dummy_model.predict(X_test_imputed)

dummy_accuracy = accuracy_score(y_test, y_pred_dummy)

print(f"Baseline Accuracy: {dummy_accuracy:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred_dummy)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Purchase", "Purchase"]
).plot()

plt.title("Dummy Classifier – Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_dummy,
        target_names=["No Purchase", "Purchase"],
        zero_division=0
    )
)

### Logistic Regression

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)

X_test_scaled = scaler.transform(X_test_imputed)

In [ ]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)

y_pred_logistic = logistic_model.predict(X_test_scaled)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_logistic,
        target_names=["No Purchase", "Purchase"],
        zero_division=0
    )
)

In [ ]:
cm_logistic = confusion_matrix(
    y_test,
    y_pred_logistic
)

ConfusionMatrixDisplay(
    confusion_matrix=cm_logistic,
    display_labels=["No Purchase", "Purchase"]
).plot()

plt.title("Logistic Regression – Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, RocCurveDisplay

y_prob_logistic = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

logistic_auc = roc_auc_score(
    y_test,
    y_prob_logistic
)

print(f"Logistic Regression ROC-AUC: {logistic_auc:.4f}")

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_prob_logistic,
    name="Logistic Regression"
)

plt.title("Logistic Regression – ROC Curve")
plt.show()

### Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_imputed, y_train)

y_pred_rf = rf_model.predict(X_test_imputed)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_rf,
        target_names=["No Purchase", "Purchase"],
        zero_division=0
    )
)

In [ ]:
y_prob_rf = rf_model.predict_proba(X_test_imputed)[:, 1]

rf_auc = roc_auc_score(y_test, y_prob_rf)

print(f"Random Forest ROC-AUC: {rf_auc:.4f}")

In [ ]:
rf_train_pred = rf_model.predict(X_train_imputed)

rf_train_accuracy = accuracy_score(y_train, rf_train_pred)
rf_test_accuracy = accuracy_score(y_test, y_pred_rf)

print(f"Train Accuracy: {rf_train_accuracy:.4f}")
print(f"Test Accuracy:  {rf_test_accuracy:.4f}")

#### Overfitting Investigation

In [ ]:
rf_limited = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)

rf_limited.fit(X_train_imputed, y_train)

train_pred_limited = rf_limited.predict(X_train_imputed)

In [ ]:
train_accuracy_limited = accuracy_score(
    y_train,
    train_pred_limited
)

print(f"Limited Forest Train Accuracy: {train_accuracy_limited:.4f}")

In [ ]:
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
rf_pipeline_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Pipeline ROC-AUC Scores:", rf_pipeline_scores)
print(f"Mean ROC-AUC: {rf_pipeline_scores.mean():.4f}")

In [ ]:
rf_default_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

rf_default_cv_scores = cross_val_score(
    rf_default_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Default Random Forest ROC-AUC:", rf_default_cv_scores)
print(f"Mean ROC-AUC: {rf_default_cv_scores.mean():.4f}")

print(f"Limited Random Forest Mean ROC-AUC: {rf_pipeline_scores.mean():.4f}")

#### Hyperparameter Tuning

In [ ]:
depth_grid = {
    "model__max_depth": [3, 5, 7, 10, None]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=depth_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best max_depth:", grid_search.best_params_)
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

In [ ]:
depth_results = pd.DataFrame(grid_search.cv_results_)

depth_comparison = depth_results[
    ["param_model__max_depth", "mean_test_score", "std_test_score"]
].copy()

depth_comparison = depth_comparison.sort_values(
    by="mean_test_score",
    ascending=False
)

depth_comparison

**Hyperparameter Tuning Results**

GridSearchCV was used to evaluate different `max_depth` values using 5-fold stratified cross-validation and ROC-AUC scoring.

Among the tested values, `max_depth=7` achieved the highest mean cross-validation ROC-AUC of **0.8497**, compared with **0.8348** for the unrestricted Random Forest.

This suggests that limiting tree depth can improve generalization. However, the improvement over `max_depth=5` was small, so the selected depth should not be considered universally optimal.

In [ ]:
parameter_grid = {
    "model__max_depth": [5, 7, 10],
    "model__min_samples_leaf": [1, 5, 10, 20]
}

rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=parameter_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid_search.fit(X_train, y_train)

print("Best Parameters:", rf_grid_search.best_params_)
print(f"Best CV ROC-AUC: {rf_grid_search.best_score_:.4f}")

#### Model Comparison Using Cross-Validation

In [ ]:
logistic_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logistic_cv_scores = cross_val_score(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression CV Scores:", logistic_cv_scores)
print(f"Logistic Regression Mean ROC-AUC: {logistic_cv_scores.mean():.4f}")
print(f"Tuned Random Forest Mean ROC-AUC: {rf_grid_search.best_score_:.4f}")

#### Final Classification Evaluation

In [ ]:
best_rf_model = rf_grid_search.best_estimator_

y_pred_rf_final = best_rf_model.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred_rf_final,
        target_names=["No Purchase", "Purchase"],
        zero_division=0
    )
)

In [ ]:
cm_rf_final = confusion_matrix(y_test, y_pred_rf_final)

ConfusionMatrixDisplay(
    confusion_matrix=cm_rf_final,
    display_labels=["No Purchase", "Purchase"]
).plot()

plt.title("Tuned Random Forest - Confusion Matrix")
plt.show()

In [ ]:
y_prob_rf_final = best_rf_model.predict_proba(X_test)[:, 1]

rf_final_auc = roc_auc_score(
    y_test,
    y_prob_rf_final
)

print(f"Tuned Random Forest Test ROC-AUC: {rf_final_auc:.4f}")

**Classification Model Evaluation Summary**

Logistic Regression and Random Forest were evaluated for predicting whether a customer would make a purchase in 2025.

Hyperparameter tuning improved the Random Forest's cross-validation ROC-AUC from 0.8348 to 0.8505.

On the test set, Logistic Regression achieved a ROC-AUC of 0.8563, while the tuned Random Forest achieved 0.8491.

At the default classification threshold of 0.50, Logistic Regression also achieved higher purchase-class recall and F1-score.

Both models demonstrated useful predictive performance, but their differences were relatively small. The choice of model and classification threshold should ultimately depend on the business objective.

The test set was inspected during model exploration, so the reported test metrics should be interpreted with that limitation in mind.

## 5. Regression Modeling

### Target Variable Analysis

In [ ]:
regression_df = df[df["future_purchase"] == 1].copy()

print("Regression Dataset Shape:", regression_df.shape)

regression_df["future_spend"].describe()

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    regression_df["future_spend"],
    bins=50,
    edgecolor="black"
)

plt.title("Distribution of Future Customer Spend")
plt.xlabel("Future Spend")
plt.ylabel("Number of Customers")

plt.show()

### Regression Data Preparation

In [ ]:
reg_train_mask = (
    y_train == 1
)

reg_test_mask = (
    y_test == 1
)

X_train_reg = X_train.loc[reg_train_mask]
X_test_reg = X_test.loc[reg_test_mask]

y_train_reg = df.loc[X_train_reg.index, "future_spend"]
y_test_reg = df.loc[X_test_reg.index, "future_spend"]

print("X_train_reg:", X_train_reg.shape)
print("X_test_reg:", X_test_reg.shape)
print("y_train_reg:", y_train_reg.shape)
print("y_test_reg:", y_test_reg.shape)

### Linear Regression

In [ ]:
linear_reg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_reg_pipeline.fit(X_train_reg, y_train_reg)

y_pred_linear = linear_reg_pipeline.predict(X_test_reg)

print("First 5 Predictions:", y_pred_linear[:5])
print("First 5 Actual Values:", y_test_reg.iloc[:5].values)

In [ ]:
linear_mae = mean_absolute_error(
    y_test_reg, y_pred_linear
)

linear_rmse = np.sqrt(
    mean_squared_error(y_test_reg, y_pred_linear)
)

linear_r2 = r2_score(
    y_test_reg, y_pred_linear
)

print(f"Linear Regression MAE: {linear_mae:,.2f}")
print(f"Linear Regression RMSE: {linear_rmse:,.2f}")
print(f"Linear Regression R²: {linear_r2:.4f}")

### Regression Baseline

In [ ]:
dummy_reg = DummyRegressor(strategy="mean")

dummy_reg.fit(X_train_reg, y_train_reg)

y_pred_dummy_reg = dummy_reg.predict(X_test_reg)

dummy_mae = mean_absolute_error(
    y_test_reg, y_pred_dummy_reg
)

dummy_rmse = np.sqrt(
    mean_squared_error(y_test_reg, y_pred_dummy_reg)
)

dummy_r2 = r2_score(
    y_test_reg, y_pred_dummy_reg
)

print(f"Dummy Regressor MAE: {dummy_mae:,.2f}")
print(f"Dummy Regressor RMSE: {dummy_rmse:,.2f}")
print(f"Dummy Regressor R²: {dummy_r2:.4f}")

### Random Forest Regression

In [ ]:
rf_reg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

rf_reg_pipeline.fit(X_train_reg, y_train_reg)

y_pred_rf_reg = rf_reg_pipeline.predict(X_test_reg)

In [ ]:
rf_reg_mae = mean_absolute_error(
    y_test_reg, y_pred_rf_reg
)

rf_reg_rmse = np.sqrt(
    mean_squared_error(y_test_reg, y_pred_rf_reg)
)

rf_reg_r2 = r2_score(
    y_test_reg, y_pred_rf_reg
)

print(f"Random Forest MAE: {rf_reg_mae:,.2f}")
print(f"Random Forest RMSE: {rf_reg_rmse:,.2f}")
print(f"Random Forest R²: {rf_reg_r2:.4f}")

In [ ]:
y_train_pred_rf_reg = rf_reg_pipeline.predict(X_train_reg)

rf_train_mae = mean_absolute_error(
    y_train_reg, y_train_pred_rf_reg
)

rf_train_r2 = r2_score(
    y_train_reg, y_train_pred_rf_reg
)

print(f"Training MAE: {rf_train_mae:,.2f}")
print(f"Test MAE: {rf_reg_mae:,.2f}")

print(f"Training R²: {rf_train_r2:.4f}")
print(f"Test R²: {rf_reg_r2:.4f}")

### Regularized Random Forest Regression

In [ ]:
rf_reg_regularized = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

rf_reg_regularized.fit(X_train_reg, y_train_reg)

y_train_pred_regularized = rf_reg_regularized.predict(X_train_reg)
y_test_pred_regularized = rf_reg_regularized.predict(X_test_reg)

print("Training Performance")
print(
    "MAE:",
    round(mean_absolute_error(y_train_reg, y_train_pred_regularized), 2)
)
print(
    "R²:",
    round(r2_score(y_train_reg, y_train_pred_regularized), 4)
)

print("\nTest Performance")
print(
    "MAE:",
    round(mean_absolute_error(y_test_reg, y_test_pred_regularized), 2)
)
print(
    "R²:",
    round(r2_score(y_test_reg, y_test_pred_regularized), 4)
)

### Log-Transformed Linear Regression

In [ ]:
log_linear_reg = TransformedTargetRegressor(
    regressor=Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

log_linear_reg.fit(X_train_reg, y_train_reg)

y_pred_log_linear = log_linear_reg.predict(X_test_reg)

log_linear_mae = mean_absolute_error(
    y_test_reg, y_pred_log_linear
)

log_linear_rmse = np.sqrt(
    mean_squared_error(y_test_reg, y_pred_log_linear)
)

log_linear_r2 = r2_score(
    y_test_reg, y_pred_log_linear
)

print(f"Log Linear MAE: {log_linear_mae:,.2f}")
print(f"Log Linear RMSE: {log_linear_rmse:,.2f}")
print(f"Log Linear R²: {log_linear_r2:.4f}")

### Regression Model Cross-Validation

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

reg_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

regression_models = {
    "Linear Regression": linear_reg_pipeline,
    "Regularized Random Forest": rf_reg_regularized,
    "Log Linear Regression": log_linear_reg
}

for model_name, model in regression_models.items():

    cv_mae = -cross_val_score(
        model,
        X_train_reg,
        y_train_reg,
        cv=reg_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    print(
        f"{model_name} CV MAE: "
        f"{cv_mae.mean():,.2f} "
        f"(Std: {cv_mae.std():,.2f})"
    )

### Regression Model Selection

Three regression approaches were compared using 5-fold cross-validation: Linear Regression, Regularized Random Forest, and Log-Transformed Linear Regression.

Log-Transformed Linear Regression achieved the lowest cross-validation MAE (57,217.65) and was selected as the conditional customer spending model.

However, its negative test R² (-0.1851) indicates limited predictive performance, particularly when larger errors are penalized.

The model was selected based on MAE, while its limitations and the exploratory use of the test set are acknowledged.

## 6. Expected Customer Value
### Purchase Probability Prediction

In [ ]:
purchase_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Number of Customers:", len(purchase_probabilities))
print("First 5 Purchase Probabilities:", purchase_probabilities[:5])

### Conditional Spending Prediction

In [ ]:
conditional_spend_predictions = log_linear_reg.predict(X_test)

conditional_spend_predictions = np.maximum(
    conditional_spend_predictions,
    0
)

print("Number of Customers:", len(conditional_spend_predictions))
print("First 5 Conditional Spend Predictions:", conditional_spend_predictions[:5])

### Expected Customer Spend Calculation

In [ ]:
expected_spend = (
    purchase_probabilities * conditional_spend_predictions
)

expected_spend_df = pd.DataFrame({
    "customer_id": df.loc[X_test.index, "customer_id"].values,
    "purchase_probability": purchase_probabilities,
    "conditional_spend": conditional_spend_predictions,
    "expected_spend": expected_spend,
    "actual_spend": df.loc[X_test.index, "future_spend"].values
})

expected_spend_df.head()

### Expected Customer Spend Evaluation

In [ ]:
expected_mae = mean_absolute_error(
    expected_spend_df["actual_spend"],
    expected_spend_df["expected_spend"]
)

expected_rmse = np.sqrt(
    mean_squared_error(
        expected_spend_df["actual_spend"],
        expected_spend_df["expected_spend"]
    )
)

expected_r2 = r2_score(
    expected_spend_df["actual_spend"],
    expected_spend_df["expected_spend"]
)

print(f"Expected Spend MAE: {expected_mae:,.2f}")
print(f"Expected Spend RMSE: {expected_rmse:,.2f}")
print(f"Expected Spend R²: {expected_r2:.4f}")

### Expected Spend Baseline Comparison

In [ ]:
baseline_spend = df.loc[
    X_train.index, "future_spend"
].mean()

baseline_predictions = np.full(
    len(X_test),
    baseline_spend
)

baseline_mae = mean_absolute_error(
    expected_spend_df["actual_spend"],
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        expected_spend_df["actual_spend"],
        baseline_predictions
    )
)

baseline_r2 = r2_score(
    expected_spend_df["actual_spend"],
    baseline_predictions
)

print(f"Baseline Mean Spend: {baseline_spend:,.2f}")
print(f"Baseline MAE: {baseline_mae:,.2f}")
print(f"Baseline RMSE: {baseline_rmse:,.2f}")
print(f"Baseline R²: {baseline_r2:.4f}")

**Expected Spend Model Evaluation Summary**

The two-stage Expected Customer Spend model combines purchase probabilities from Logistic Regression with conditional spending predictions from Log-Transformed Linear Regression.

The model achieved a test MAE of 20,970.13, compared with 30,589.39 for the mean-spend baseline, representing an approximately 31.45% reduction in MAE.

However, the improvement in RMSE was relatively small, and the model's R² remained low (0.0362). This suggests that predicting high-spending customers accurately remains challenging.

The results indicate potential value for customer-level spending estimates and prioritization, but further validation is required before operational use.

The test set was inspected during model development, so these results should be interpreted as exploratory rather than fully independent validation.

### Aggregate Spend Evaluation

In [ ]:
actual_total_spend = expected_spend_df["actual_spend"].sum()

predicted_total_spend = expected_spend_df["expected_spend"].sum()

total_spend_error_pct = (
    (predicted_total_spend - actual_total_spend)
    / actual_total_spend
) * 100

print(f"Actual Total Spend: {actual_total_spend:,.2f}")
print(f"Predicted Total Spend: {predicted_total_spend:,.2f}")
print(f"Total Spend Error (%): {total_spend_error_pct:.2f}%")

### Spend Prediction by Customer Group

In [ ]:
spend_analysis_df = expected_spend_df.copy()

spend_analysis_df["spend_group"] = pd.cut(
    spend_analysis_df["actual_spend"],
    bins=[-1, 0, 10000, 50000, 100000, float("inf")],
    labels=[
        "No Purchase",
        "0-10K",
        "10K-50K",
        "50K-100K",
        "100K+"
    ]
)

spend_group_summary = (
    spend_analysis_df
    .groupby("spend_group", observed=True)
    .agg(
        customer_count=("customer_id", "count"),
        actual_total=("actual_spend", "sum"),
        predicted_total=("expected_spend", "sum")
    )
)

spend_group_summary["prediction_error"] = (
    spend_group_summary["predicted_total"]
    - spend_group_summary["actual_total"]
)

print(spend_group_summary.round(2))

### Expected Spend Regression Model Comparison

In [ ]:
# Conditional spend predictions from Linear Regression
linear_spend_predictions = np.maximum(
    linear_reg_pipeline.predict(X_test),
    0
)

# Expected spend using Linear Regression
expected_spend_linear = (
    purchase_probabilities * linear_spend_predictions
)

# Expected spend using Log Linear Regression
expected_spend_log_linear = expected_spend_df["expected_spend"].values

# Actual spending
actual_spend = expected_spend_df["actual_spend"].values

# Compare both approaches
comparison_results = pd.DataFrame({
    "Model": ["Linear Regression", "Log Linear Regression"],
    "MAE": [
        mean_absolute_error(actual_spend, expected_spend_linear),
        mean_absolute_error(actual_spend, expected_spend_log_linear)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(actual_spend, expected_spend_linear)),
        np.sqrt(mean_squared_error(actual_spend, expected_spend_log_linear))
    ],
    "Predicted Total Spend": [
        expected_spend_linear.sum(),
        expected_spend_log_linear.sum()
    ]
})

print(comparison_results.round(2))
print(f"\nActual Total Spend: {actual_spend.sum():,.2f}")

### Expected Customer Spend – Final Findings

The Expected Customer Spend analysis combined purchase probability predictions from Logistic Regression with conditional spending estimates from regression models.

Two regression approaches were compared: Linear Regression and Log-Transformed Linear Regression.

Log-Transformed Linear Regression achieved a lower MAE, indicating better average absolute prediction accuracy. However, Linear Regression achieved a lower RMSE and produced aggregate spending estimates closer to the actual total.

The analysis also revealed substantial prediction errors among high-spending customers, highlighting the difficulty of modeling highly skewed customer spending distributions.

Linear Regression was selected as the provisional conditional spending model for the two-stage Expected Customer Spend framework, based on its aggregate spending performance and lower RMSE.

The findings are exploratory, and independent validation would be required before deploying the model in a real business environment.

### Export Customer Prediction Results

In [ ]:
customer_prediction_results = pd.DataFrame({
    "customer_id": df.loc[X_test.index, "customer_id"].values,
    "purchase_probability": purchase_probabilities,
    "predicted_conditional_spend": np.maximum(
        linear_reg_pipeline.predict(X_test), 0
    ),
    "actual_purchase": df.loc[X_test.index, "future_purchase"].values,
    "actual_spend": df.loc[X_test.index, "future_spend"].values
})

customer_prediction_results["expected_spend"] = (
    customer_prediction_results["purchase_probability"]
    * customer_prediction_results["predicted_conditional_spend"]
)

print(customer_prediction_results.shape)
print(customer_prediction_results.head())

### Full Customer Prediction Results

In [ ]:
full_customer_predictions = pd.DataFrame({
    "customer_id": df["customer_id"].values,
    "purchase_probability": logistic_pipeline.predict_proba(X)[:, 1],
    "predicted_conditional_spend": np.maximum(
        linear_reg_pipeline.predict(X), 0
    )
})

full_customer_predictions["expected_spend"] = (
    full_customer_predictions["purchase_probability"]
    * full_customer_predictions["predicted_conditional_spend"]
)

print(full_customer_predictions.shape)
print(full_customer_predictions.head())

In [ ]:
from pathlib import Path

output_dir = Path("../../04_power_bi/data")
output_dir.mkdir(parents=True, exist_ok=True)

full_customer_predictions.to_csv(
    output_dir / "customer_predictions.csv",
    index=False
)

customer_prediction_results.to_csv(
    output_dir / "customer_predictions_test.csv",
    index=False
)

print("Prediction files exported successfully.")